# FaceSwap Studio - Kaggle T4 x2 Dual-GPU Launcher

This notebook launches **FaceSwap Studio** on Kaggle's dual NVIDIA T4 GPUs (`T4 x2`).
It initializes the runtime environment, prefetches FaceFusion v3 model weights, starts the asynchronous FastAPI backend server, and establishes a secure Cloudflare Quick Tunnel (`*.trycloudflare.com`) for connecting your browser studio UI.

> Note: Ensure Kaggle session accelerator is configured with **GPU T4 x2** and **Internet access is toggled ON** in Notebook settings.

### Step 0: Repository Clone & Working Directory Setup
Verifies internet connectivity and clones the FaceSwap Studio repository into `/kaggle/working/faceswap-studio`.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/selva-1201/faceswap-studio.git"
REPO_DIR = "/kaggle/working/faceswap-studio"

# 1. Verify Internet connectivity
try:
    res = subprocess.run(["curl", "-sI", "--max-time", "5", "https://github.com"], capture_output=True, text=True)
    if "HTTP/" not in res.stdout:
        print("[Error] Internet access is not enabled in this Kaggle session! Go to Notebook Settings -> Internet -> ON.")
    else:
        print("[OK] Internet connection verified.")
except Exception as e:
    print(f"[Error] Internet check failed: {e}")

# 2. Clone repository if not present
if not os.path.exists(REPO_DIR):
    print(f"[Git] Cloning {REPO_URL} into {REPO_DIR}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print(f"[Git] Repository already present at {REPO_DIR}.")

os.chdir(REPO_DIR)
print("[Workspace] Current Working Directory:", os.getcwd())

### Step 1: Verify Hardware & Dual-GPU Visibility
Inspects GPU hardware using `nvidia-smi -L` without initializing CUDA or PyTorch contexts.

In [ ]:
import subprocess

try:
    res = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, check=True)
    gpu_lines = [line.strip() for line in res.stdout.strip().split("\n") if line.strip()]
    gpu_count = len(gpu_lines)
    print(f"[Hardware] Detected {gpu_count} GPU device(s):")
    for line in gpu_lines:
        print(" ", line)

    if gpu_count < 2:
        print("\n[Warning] Dual GPUs (T4 x2) not detected. Pipeline will run but parallel dual-GPU chunking requires 2 devices.")
    else:
        print("\n[OK] Dual-GPU environment verified and ready for parallel execution.")
except Exception as e:
    print(f"[Notice] nvidia-smi failed or not available: {e}")

### Step 2: Environment Setup & Model Weight Prefetching
Installs dependencies with Kaggle package protection, sets up FaceFusion v3, and prefetches model weights into `/kaggle/working/facefusion/.assets/models`.

In [ ]:
import subprocess
import sys

REPO_DIR = "/kaggle/working/faceswap-studio"
setup_script = f"{REPO_DIR}/scripts/setup_env.sh"
prefetch_script = f"{REPO_DIR}/scripts/prefetch_models.py"

print("[Setup] Executing environment initialization...")
proc_setup = subprocess.Popen(["bash", setup_script], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in iter(proc_setup.stdout.readline, ""):
    sys.stdout.write(line)
    sys.stdout.flush()
proc_setup.wait()
if proc_setup.returncode != 0:
    raise RuntimeError(f"setup_env.sh failed with exit code {proc_setup.returncode}")

print("\n[Prefetch] Prefetching model weights...")
proc_prefetch = subprocess.Popen(["python3", prefetch_script], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in iter(proc_prefetch.stdout.readline, ""):
    sys.stdout.write(line)
    sys.stdout.flush()
proc_prefetch.wait()
if proc_prefetch.returncode != 0:
    print(f"[Warning] prefetch_models.py exited with code {proc_prefetch.returncode}. Missing models will auto-download on first job.")

### Step 3: Launch FastAPI Background Server
Starts the non-blocking job manager server on port `7860` in its own detached process session (`start_new_session=True`).

In [ ]:
import os
import subprocess
import time
import requests

REPO_DIR = "/kaggle/working/faceswap-studio"
SERVER_LOG = "/kaggle/working/server.log"

def start_server():
    os.system('pkill -f "backend.server" 2>/dev/null || true')
    time.sleep(1)

    env = os.environ.copy()
    env["PYTHONPATH"] = f"{REPO_DIR}:{env.get('PYTHONPATH', '')}"

    print("[Server] Starting FaceSwap Studio backend on port 7860...")
    proc = subprocess.Popen(
        ["python3", "-m", "backend.server"],
        cwd=REPO_DIR,
        env=env,
        stdout=open(SERVER_LOG, "w"),
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

    # Wait for server to become responsive
    max_retries = 30
    for _ in range(max_retries):
        try:
            res = requests.get("http://127.0.0.1:7860/api/system/status", timeout=2)
            if res.status_code == 200:
                print("[Server] Backend API is up and running!")
                print(res.json())
                return proc
        except Exception:
            time.sleep(1)

    print("[Error] Server failed to respond within 30 seconds. Showing recent log output:")
    if os.path.exists(SERVER_LOG):
        with open(SERVER_LOG, "r") as f:
            print(f.read()[-2000:])
    raise RuntimeError("Backend server startup timed out.")

server_process = start_server()

### Step 3b: Server Restart Helper (Run when updating code)
Quickly pulls latest changes and restarts the backend server cleanly.

In [ ]:
def restart_server():
    global server_process
    print("[Server] Restarting server instance...")
    server_process = start_server()

# To restart after git pull, uncomment and execute:
# !cd /kaggle/working/faceswap-studio && git pull origin main
# restart_server()

### Step 4: Launch Cloudflare Quick Tunnel
Exposes port `7860` to the internet via Cloudflare Quick Tunnel (`start_new_session=True`).
> Important: Cloudflare Quick Tunnel has an HTTP request body size limit of **100MB**. FaceSwap Studio enforces a maximum video upload of **95MB**.

In [ ]:
import os
import re
import subprocess
import time
from IPython.display import HTML, display

tunnel_bin = "/kaggle/working/cloudflared"
tunnel_log = "/kaggle/working/tunnel.log"

os.system('pkill -f "cloudflared tunnel" 2>/dev/null || true')
time.sleep(1)

print("[Tunnel] Spawning Cloudflare Quick Tunnel (no-autoupdate)...")
tunnel_process = subprocess.Popen(
    [tunnel_bin, "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:7860"],
    stdout=open(tunnel_log, "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

public_url = None
url_pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")

for _ in range(40):
    time.sleep(1)
    try:
        if os.path.exists(tunnel_log):
            with open(tunnel_log, "r") as f:
                match = url_pattern.search(f.read())
                if match:
                    public_url = match.group(0)
                    break
    except Exception:
        pass

if public_url:
    card_html = f"""
    <div style="background: #0b0e14; border: 1px solid #00f0ff; border-radius: 12px; padding: 20px; color: #ffffff; font-family: sans-serif; box-shadow: 0 0 20px rgba(0, 240, 255, 0.2);">
      <h2 style="margin: 0 0 10px 0; color: #00f0ff;">FaceSwap Studio Remote Tunnel Ready</h2>
      <p style="margin: 0 0 14px 0; color: #8e9bb0;">Open the link below or paste into Remote Backend URL in your local studio UI (max upload 95MB):</p>
      <div style="margin-bottom: 12px;">
        <a href="{public_url}" target="_blank" style="background: linear-gradient(135deg, #00c6ff, #0072ff); color: white; padding: 10px 20px; border-radius: 6px; text-decoration: none; font-weight: bold; display: inline-block;">
          Open FaceSwap Studio Web UI
        </a>
      </div>
      <div style="font-family: monospace; background: #040507; padding: 8px 12px; border-radius: 6px; border: 1px solid rgba(255,255,255,0.1); color: #00ff88; word-break: break-all;">
        {public_url}
      </div>
    </div>
    """
    display(HTML(card_html))
else:
    print("[Warning] Cloudflare URL not detected within 40 seconds. Check /kaggle/working/tunnel.log:")
    if os.path.exists(tunnel_log):
        with open(tunnel_log, "r") as f:
            print(f.read())

### Diagnostics Cell
Checks system RAM, disk spaces, GPU allocation, process statuses, and recent server logs.

In [ ]:
import subprocess
import os

print("=== SYSTEM MEMORY ===")
subprocess.run(["free", "-h"])

print("\n=== DISK SPACE ===")
subprocess.run(["df", "-h", "/kaggle/working", "/tmp"])

print("\n=== GPU STATUS ===")
subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total,memory.used,memory.free", "--format=csv"])

print("\n=== PROCESS STATUS ===")
server_alive = (server_process.poll() is None) if 'server_process' in globals() else False
tunnel_alive = (tunnel_process.poll() is None) if 'tunnel_process' in globals() else False
print(f"FastAPI Server running: {server_alive}")
print(f"Cloudflare Tunnel running: {tunnel_alive}")

print("\n=== RECENT SERVER LOGS (Last 60 lines) ===")
if os.path.exists("/kaggle/working/server.log"):
    with open("/kaggle/working/server.log", "r") as f:
        lines = f.readlines()
        print("".join(lines[-60:]))
else:
    print("No server.log file found.")

### Step 5: Keep-Alive & Job Heartbeat Loop
Runs an active monitoring loop displaying live RAM, disk, and GPU memory metrics every 10s.
If the server dies, checks kernel OOM logs (`dmesg`) and auto-restarts up to 3 times before stopping.

In [ ]:
import time
import requests
import subprocess
from datetime import datetime

restart_attempts = 0
MAX_RESTARTS = 3

print("[Monitor] Keep-alive monitoring loop started. Press Stop in Kaggle to terminate.")
while True:
    try:
        # Check resources
        res = requests.get("http://127.0.0.1:7860/api/system/resources", timeout=5)
        if res.status_code == 200:
            data = res.json()
            ram = data.get("ram", {})
            disk_w = data.get("disk_workspace", {})
            gpus = data.get("gpus", [])
            gpu_str = " | ".join([f"GPU {g['index']}: {g['used_mb']:.0f}/{g['total_mb']:.0f}MB" for g in gpus])
            ts = datetime.now().strftime("%H:%M:%S")
            print(f"[{ts}] RAM: {ram.get('available_mb', 0):.0f}MB free | Disk: {disk_w.get('free_gb', 0):.1f}GB free | {gpu_str}", end="\r")
    except Exception:
        # Server unresponsive - check process health
        if 'server_process' in globals() and server_process.poll() is not None:
            print(f"\n[Alert] Backend server exited with code {server_process.returncode}!")
            if restart_attempts < MAX_RESTARTS:
                restart_attempts += 1
                print(f"[Recovery] Attempting server auto-restart ({restart_attempts}/{MAX_RESTARTS})...")
                # Print dmesg OOM hints
                try:
                    dmesg_res = subprocess.run(["dmesg", "-T"], capture_output=True, text=True)
                    oom_lines = [l for l in dmesg_res.stdout.split("\n") if any(k in l.lower() for k in ["oom", "kill", "out of memory")][-10:]
                    if oom_lines:
                        print("[Kernel OOM Trace]:")
                        for l in oom_lines:
                            print("  ", l)
                except Exception:
                    pass
                server_process = start_server()
            else:
                print("[Fatal] Maximum restart attempts reached. Halting keep-alive loop.")
                break

    time.sleep(10)